# Snowflake x GenAI: LLM Functions: Lesson 3: The SENTIMENT() Function

SENTIMENT, the API Ninjas quotes function, RAW_QUOTES.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `SXGA2_LLM_ADMIN` (ACCOUNTADMIN), `SXGA2_LLM_ROLE` (SYSADMIN) and `SXGA2_LLM_WH` (COMPUTE_WH).
- Course code that no longer works runs **as written** and reports its error; a **Substitute** cell with current functions follows it.
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- 🛠️ **Course update needed** marks what has to change in the course itself.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "SXGA2_LLM_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "SXGA2_LLM_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "SXGA2_LLM_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: It's a Whole Mood

> **Course:** 🧪 LAB: Plug a Mood Into the SENTIMENT() Function; try the other moods (Giddy, Reflective, Pensive, Agitated, Blithe, Rhapsodic, Dejected, Doleful, Lacrymose). Troubleshooting: set the worksheet context to SNOWFLAKE.CORTEX or fully qualify the name.

> ⚠️ **Audit note:** Substitute cells use the current `AI_*` functions. The legacy `SNOWFLAKE.CORTEX.TRANSLATE`, `SENTIMENT` and `SUMMARIZE` are blocked on new accounts (verified 2026-11-14: "New accounts are blocked from using legacy SNOWFLAKE.CORTEX functions").

In [ ]:
%%sql -r sentiment_as_given
execute immediate 'begin
  execute immediate ''select sentiment(''''disconsolate'''')'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

In [ ]:
%%sql -r sentiment_qualified_as_given
execute immediate 'begin
  execute immediate ''select snowflake.cortex.sentiment(''''disconsolate'''')'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

In [ ]:
%%sql -r sentiment_substitute
-- Substitute: AI_SENTIMENT returns an OBJECT with a label per category, not a number from -1 to 1
select ai_sentiment('disconsolate') as result,
       ai_sentiment('disconsolate'):categories[0]:sentiment::text as label;

In [ ]:
%%sql -r sentiment_moods_substitute
select mood, ai_sentiment(mood):categories[0]:sentiment::text as label
from (select column1 as mood from values ('Giddy'),('Reflective'),('Pensive'),('Agitated'),('Blithe'),('Rhapsodic'),('Dejected'),('Doleful'),('Lacrymose'));

> 🛠️ **Course update needed:** `AI_SENTIMENT` is not a drop-in replacement: it returns labels (positive / negative / neutral / mixed / unknown), not a score. The page premise ("ranks it between -1 and 1"), the lab prediction exercise, LLMM_090 and everything later that uses a score must be rewritten.

## Page: Cyborg Bear Animation - Round 2

Reading and quiz only.

## Page: Applying Sentiment to Quotes

> **Course:** 🧪 LAB: Sign Up for a Free Account on API Ninjas (LLMM_091-095): verify your email, log in to the dashboard, copy the API key ("the API Key is your password for using the API").

In [ ]:
%%sql -r secret_exists
-- Not a course cell: confirms the API key secret from the setup notebook exists (the key itself is never shown)
show secrets like 'SXGA2_API_NINJAS_KEY' in schema UTIL_DB.PUBLIC;

## Page: Preparing Code to Call the API

> **Course:** 🧪 LAB: Reading the Documentation (api-ninjas.com quotes API: optional parameter `category`; the key goes in the `X-Api-Key` header). 🧪 LAB: paste the sample Python into a worksheet, create a variable for the API key, convert PRINT() to RETURN, add the function header and closing `$$`s (LLMM_098, 100, 101).

## Page: Finessing Our Function that Calls the API

> **Course:** 🎯 CHALLENGE LAB: Create an SXGA2 Infrastructure: give SYSADMIN USAGE on COMPUTE_WH; set your default role to SYSADMIN; set your worksheet role to SYSADMIN; create database SXGA2; drop the PUBLIC schema; create schema NINJAS; double-check both are owned by SYSADMIN.

In [ ]:
%%sql -r create_sxga2
-- Derived solution. The stand-in warehouse is already owned by the SYSADMIN stand-in, and default role/user settings are not changed here.
use role {{SYSADMIN_ROLE}};
create database SXGA2;
drop schema SXGA2.PUBLIC;
create schema SXGA2.NINJAS;
use warehouse {{WAREHOUSE}};
use schema SXGA2.NINJAS;

In [ ]:
%%sql -r check_sxga2_owner
show databases like 'SXGA2';
show schemas in database SXGA2;

> **Course:** 🧪 LAB: Try to Create the Function (LLMM_101, 102): the first version fails with `'return' outside function`.

> ⚠️ **Audit note:** The course's functions use `runtime_version=3.9`, which no longer exists ("Python runtime version 3.9 is decommissioned"). They run as written below and report that error; the **Substitute** cell uses 3.12.

In [ ]:
%%sql -r function_attempt_1
execute immediate 'begin
  execute immediate ''create or replace function get_api_data()
returns varchar
language python
runtime_version=3.9
handler = ''''get_the_data''''
packages = (''''requests'''')
as
$$
import requests

api_url = ''''https://api.api-ninjas.com/v1/quotes''''
api_key = ''''YOUR API KEY''''

response = requests.get(api_url, headers={''''X-Api-Key'''': api_key})
if response.status_code == requests.codes.ok:
    return response.text
else:
    return "Error:", response.status_code, response.text
$$'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

> **Course:** 🧪 LAB: Try to Create the Function, Again: add a line and indent the lines underneath to create the HANDLER (LLMM_103), then try to call it (LLMM_104).

In [ ]:
%%sql -r function_attempt_2
execute immediate 'begin
  execute immediate ''create or replace function get_api_data()
returns varchar
language python
runtime_version=3.9
handler = ''''get_the_data''''
packages = (''''requests'''')
as
$$
import requests

api_url = ''''https://api.api-ninjas.com/v1/quotes''''
api_key = ''''YOUR API KEY''''

def get_the_data():
    response = requests.get(api_url, headers={''''X-Api-Key'''': api_key})
    if response.status_code == requests.codes.ok:
        return response.text
    else:
        return response.status_code
$$'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

In [ ]:
%%sql -r call_function_attempt_2
execute immediate 'begin
  execute immediate ''select get_api_data()'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

> **Course:** 🧪 LAB: Create a Network Rule (LLMM_106, 107): name **api_ninjas_network_access** in SXGA2.NINJAS, comment *this makes it okay for us to send messages out to the api*, Type Host Port, Mode Egress, identifier **api.api-ninjas.com**.

In [ ]:
%%sql -r create_network_rule
-- SQL equivalent of the Create Network Rule dialog (LLMM_106)
use role {{ADMIN_ROLE}};
use warehouse {{WAREHOUSE}};
use schema SXGA2.NINJAS;
create or replace network rule api_ninjas_network_access
  mode = egress
  type = host_port
  value_list = ('api.api-ninjas.com')
  comment = 'this makes it okay for us to send messages out to the api';

> **Course:** 🧪 LAB: Put the Rule into an Integration. *If you are using a trial account you MUST enter a billing credit card.*

In [ ]:
%%sql -r create_integration
use schema SXGA2.NINJAS;
use role {{ADMIN_ROLE}};
create or replace EXTERNAL ACCESS INTEGRATION api_ninjas_access_integration
 allowed_network_rules = (api_ninjas_network_access)
 enabled = true;

In [ ]:
%%sql -r grant_integration
grant usage on integration api_ninjas_access_integration to role {{SYSADMIN_ROLE}};

> 🛠️ **Course update needed:** The note that a credit card is required for external access on a trial is wrong according to the existing audit: trial accounts cannot use external network access at all, with or without a card.

> **Course:** 🧪 LAB: Add the Integration to the Function (then test it once again) (LLMM_109: `external_access_integrations = (api_ninjas_access_integration)`).

In [ ]:
%%sql -r function_with_integration
execute immediate 'begin
  execute immediate ''create or replace function get_api_data()
returns varchar
language python
runtime_version=3.9
handler = ''''get_the_data''''
packages = (''''requests'''')
external_access_integrations = (api_ninjas_access_integration)
as
$$
import requests

api_url = ''''https://api.api-ninjas.com/v1/quotes''''
api_key = ''''YOUR API KEY''''

def get_the_data():
    response = requests.get(api_url, headers={''''X-Api-Key'''': api_key})
    if response.status_code == requests.codes.ok:
        return response.text
    else:
        return response.status_code
$$'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

> ⚠️ **Audit note:** **Secret instead of a typed key.** The course puts the key in the function text (`api_key = 'YOUR API KEY'`) and says later that production code should move it into a SECRET. The substitute below does that from the start so the key never appears in a notebook or the public repo: the integration is allowed to use the secret, and the function reads it with `_snowflake.get_generic_secret_string`.

In [ ]:
%%sql -r allow_secret_on_integration
-- Substitute step: let the integration use the secret
use role {{ADMIN_ROLE}};
alter external access integration api_ninjas_access_integration set allowed_authentication_secrets = (UTIL_DB.PUBLIC.SXGA2_API_NINJAS_KEY);
grant usage on integration api_ninjas_access_integration to role {{SYSADMIN_ROLE}};

In [ ]:
%%sql -r function_substitute
use role {{SYSADMIN_ROLE}};
use warehouse {{WAREHOUSE}};
use schema SXGA2.NINJAS;
-- Substitute: Python 3.12 and the key from a secret
create or replace function get_api_data()
returns varchar
language python
runtime_version=3.12
handler = 'get_the_data'
packages = ('requests')
external_access_integrations = (api_ninjas_access_integration)
secrets = ('api_key' = UTIL_DB.PUBLIC.SXGA2_API_NINJAS_KEY)
as
$$
import requests
import _snowflake

api_url = 'https://api.api-ninjas.com/v1/quotes'

def get_the_data():
    api_key = _snowflake.get_generic_secret_string('api_key')
    response = requests.get(api_url, headers={'X-Api-Key': api_key})
    if response.status_code == requests.codes.ok:
        return response.text
    else:
        return response.status_code
$$;

In [ ]:
%%sql -r call_function
select get_api_data();

## Page: Improving & Using Our Request Function

> **Course:** 🎯 CHALLENGE LAB: Create a Table to Hold the JSON Quotes: RAW_QUOTES with one column JSON of type VARIANT, owned by SYSADMIN.

In [ ]:
%%sql -r create_raw_quotes
-- Derived solution
create table raw_quotes (json variant);

> **Course:** 🧪 LAB: Testing Code Until We Get Results We Can Use. Run each statement separately.

In [ ]:
%%sql -r quotes_insert_raw
execute immediate 'begin
  execute immediate ''--this will fail because the data is not the right type
 insert into raw_quotes
 select get_api_data()'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

In [ ]:
%%sql -r quotes_insert_parsed
--this will succeed 
 insert into raw_quotes
 select parse_json(get_api_data());

In [ ]:
%%sql -r quotes_select_1
--the data looks okay, but having the [ brackets ] will keep us from parsing it back out, so we need to remove them
 select * from raw_quotes;

In [ ]:
%%sql -r quotes_insert_replaced
--remove the [ and ] using replace
insert into raw_quotes
 select parse_json(replace(replace(get_api_data(),'['),']'));

In [ ]:
%%sql -r quotes_select_2
--notice the different between the rows  
select * from raw_quotes;

In [ ]:
%%sql -r quotes_parse
--now to parse each value into a different column, you need the outer element (the brackets) removed
select json:author::text as author
     , json:quote::text as quote
     , json:category::text as category
from raw_quotes;

> **Course:** 🎯 CHALLENGE LAB: Generate at Least 10 Rows of Data. Make sure the rows parse properly; get rid of any rows that won't work (or edit them).

In [ ]:
%%sql -r generate_quotes
-- Derived solution: call the API nine more times with the bracket-free insert (the API returns a random quote each call)
execute immediate $$
begin
  for i in 1 to 9 do
    insert into raw_quotes select parse_json(replace(replace(get_api_data(),'['),']'));
  end for;
  return 'inserted';
end;
$$;

In [ ]:
%%sql -r remove_unparseable_rows
-- Derived solution: the rows from the first two test inserts still have the [ ] array around the quote
delete from raw_quotes where is_array(json) or json:quote is null;
select count(*) as usable_rows from raw_quotes;

In [ ]:
%%sql -r quotes_parsed_check
select json:author::text as author, json:quote::text as quote, json:category::text as category from raw_quotes;

> **Course:** 🧪 LAB: Use TRANSLATE() and SENTIMENT() on Your Quotes Table (LLMM_114).

In [ ]:
%%sql -r quotes_cortex_as_given
execute immediate 'begin
  execute immediate ''select json:author::text as author
     , snowflake.cortex.sentiment(json:quote::text) as quote_sentiment
     , snowflake.cortex.translate(json:quote::text,''''en'''', ''''sv'''') as quote_swedish
     , json:quote::text as original_quote
     , json:category::text as category
from raw_quotes'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

In [ ]:
%%sql -r quotes_cortex_substitute
-- Substitute: ai_sentiment (label) and ai_translate
select json:author::text as author
     , ai_sentiment(json:quote::text):categories[0]:sentiment::text as quote_sentiment
     , ai_translate(json:quote::text,'en', 'sv') as quote_swedish
     , json:quote::text as original_quote
     , json:category::text as category
from raw_quotes;

> ⚠️ **Audit note:** Another Improvement? The course defers moving the API key into a SECRET; this notebook already does it (see above).